# AutoLyap: Iteration-dependent Lyapunov analyses - Example

[Open in Colab](https://colab.research.google.com/github/ManuUpadhyaya/tutorial_ELLIIT_2026/blob/main/notebooks/iteration_dependent_lyapunov_example.ipynb)

This notebook reproduces the Nesterov fast-gradient example from the iteration-dependent Lyapunov analysis slides. It is fully filled in: there are no exercise placeholders.


## Setup

Install AutoLyap. The notebook uses `backend="cvxpy"` and `cvxpy_solver="CLARABEL"`, which is suitable for Google Colab and does not require a commercial SDP solver.


In [ ]:
%pip install -q autolyap


## Problem setup

Consider the smooth convex minimization problem

$$
\underset{y\in\mathcal{H}}{\operatorname{minimize}}\quad f(y),
$$

or equivalently the inclusion problem

$$
\text{find } y\in\mathcal{H}\;\text{ such that }\;0\in\partial f(y)=\{\nabla f(y)\},
$$

where

$$
f\in\mathcal{F}_{0,L}(\mathcal{H}),\qquad 0<L<+\infty.
$$

Nesterov's fast gradient method is

$$
\begin{aligned}
y^k &= x^k+\alpha_k(x^k-x^{k-1}),\\
x^{k+1} &= y^k-\gamma\nabla f(y^k),
\end{aligned}
\qquad
\begin{aligned}
\alpha_k &= \frac{\lambda_k-1}{\lambda_{k+1}},\\
\lambda_{k+1} &= \frac{1+\sqrt{1+4\lambda_k^2}}{2},
\end{aligned}
$$

with $\lambda_0=1$ and $\gamma=1/L$. We search for the smallest $c_K\geq 0$ such that

$$
f(x^K)-f(y^\star)\leq c_K\|x^0-y^\star\|^2,
\qquad
 y^\star\in\operatorname{Argmin}_{y\in\mathcal{H}} f(y).
$$


## State-space representation

$$
\begin{aligned}
\mathbf{x}^{k+1}
&=\left(\begin{bmatrix}1+\alpha_k&-\alpha_k\\1&0\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}-\gamma&0\\0&0\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{y}^{k}
&=\left(\begin{bmatrix}1+\alpha_k&-\alpha_k\\1&0\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}0&0\\0&0\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{u}^k
&\in \partial f(y_1^k)\times\partial f(y_2^k).
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbf{x}^k &= (x^k,x^{k-1}),\\
\mathbf{u}^k &= (u_1^k,u_2^k)=(\nabla f(y^k),\nabla f(x^k)),\\
\mathbf{y}^k &= (y_1^k,y_2^k)=(y^k,x^k).
\end{aligned}
$$

The built-in `NesterovFastGradientMethod` class in AutoLyap implements exactly these matrices. The second gradient evaluation is included so that the endpoint performance measure can use $x^K$ directly.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from autolyap import IterationDependent
from autolyap import SolverOptions
from autolyap.algorithms import NesterovFastGradientMethod
from autolyap.problemclass import InclusionProblem, SmoothConvex

warnings.filterwarnings(
    "ignore",
    message="Solution may be inaccurate.*",
)


## Define the problem data

The slide uses $L=1$ and therefore $\gamma=1/L=1$.


In [ ]:
L = 1.0
gamma = 1.0 / L

problem = InclusionProblem([SmoothConvex(L=L)])
algorithm = NesterovFastGradientMethod(gamma=gamma)
solver_options = SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL")


## Endpoint Lyapunov values

For iteration-dependent analyses, AutoLyap searches for intermediate Lyapunov functions so that

$$
\mathcal{V}(K)\leq\mathcal{V}(K-1)\leq\cdots\leq\mathcal{V}(1)\leq c_K\mathcal{V}(0).
$$

Here the endpoint helpers choose

$$
\mathcal{V}(0)=\|x^0-y^\star\|^2,
\qquad
\mathcal{V}(K)=f(x^K)-f(y^\star).
$$

Since $\mathbf{y}^k=(y^k,x^k)$, the state component $x^0$ and the endpoint $x^K$ are selected with `j=2`.


In [ ]:
def search_rate_for_K(K: int):
    Q_0, q_0 = IterationDependent.get_parameters_distance_to_solution(
        algorithm,
        k=0,
        i=1,
        j=2,
    )
    Q_K, q_K = IterationDependent.get_parameters_function_value_suboptimality(
        algorithm,
        k=K,
        j=2,
    )
    result = IterationDependent.search_lyapunov(
        problem,
        algorithm,
        K,
        Q_0,
        Q_K,
        q_0=q_0,
        q_K=q_K,
        solver_options=solver_options,
        verbosity=0,
    )
    if result["status"] != "feasible":
        raise RuntimeError(
            f"No feasible certificate for K={K}: "
            f"status={result['status']}, solve_status={result['solve_status']}"
        )
    return result


K = 10
single_result = search_rate_for_K(K)
print(f"K: {K}")
print(f"status: {single_result['status']}")
print(f"solver status: {single_result['solve_status']}")
print(f"c_K: {single_result['c_K']:.8g}")


## Comparison bounds

For comparison with the AutoLyap certificates, we plot the two standard rates used in the slides:

$$
\frac{L}{2\lambda_K^2},
\qquad
\frac{2L}{(K+2)^2}.
$$


In [ ]:
def lambda_value(K: int) -> float:
    lam = 1.0
    for _ in range(K):
        lam = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * lam**2))
    return lam


def nesterov_second_bound(K_values):
    lambdas = np.array([lambda_value(int(K)) for K in K_values], dtype=float)
    return L / (2.0 * lambdas**2)


def nesterov_first_bound(K_values):
    K_values = np.asarray(K_values, dtype=float)
    return 2.0 * L / (K_values + 2.0) ** 2


## Sweep over horizons

The slide plots $K=1,\ldots,100$. Each point below is a separate iteration-dependent SDP, so the cell prints progress as it runs.


In [ ]:
K_max = 100
K_values = np.arange(1, K_max + 1)

c_autolyap_values = []
solve_statuses = []

for idx, K_value in enumerate(K_values, start=1):
    result = search_rate_for_K(int(K_value))
    c_autolyap_values.append(result["c_K"])
    solve_statuses.append(result["solve_status"])
    print(
        f"K {idx:03d}/{K_max}: c_K={result['c_K']:.6e}, "
        f"solver={result['solve_status']}",
        flush=True,
    )

c_autolyap_values = np.array(c_autolyap_values, dtype=float)
print(f"computed {len(c_autolyap_values)} certificates")


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.0), dpi=160, constrained_layout=True)

ax.loglog(
    K_values,
    c_autolyap_values,
    linewidth=2.0,
    color="#4f8bc9",
    label="AutoLyap",
)
ax.loglog(
    K_values,
    nesterov_second_bound(K_values),
    linewidth=2.0,
    linestyle=(0, (4, 2)),
    color="#6f5aa7",
    label=r"$L/(2\lambda_K^2)$",
)
ax.loglog(
    K_values,
    nesterov_first_bound(K_values),
    linewidth=2.0,
    linestyle="--",
    color="#c8a437",
    label=r"$2L/(K+2)^2$",
)

ax.set_xlabel(r"$K$")
ax.set_ylabel(r"$c_K$", rotation=0)
ax.set_title(r"Nesterov fast gradient method: certified rate")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
plt.show()
